<a href="https://colab.research.google.com/github/Abrodolph/research/blob/main/grpo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Setup
Install dependencies and load the base model with LoRA adapters.

In [ ]:
# Pin TRL to avoid mergekit/llm_blender eager-import issues in newer versions
!pip install pydantic numpy imageio matplotlib -q
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
!pip install -U "trl" "transformers>=4.40" "accelerate" "datasets" "pyarrow" -q


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 42.7 MB

In [ ]:
import os, sys
if not os.path.exists('Wildfire-Containment-Simulator'):
    !git clone https://github.com/Abrodolph/Wildfire-Containment-Simulator.git
sys.path.insert(0, 'Wildfire-Containment-Simulator')
os.chdir('Wildfire-Containment-Simulator')

Cloning into 'Wildfire-Containment-Simulator'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 212 (delta 89), reused 179 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (212/212), 371.77 KiB | 7.59 MiB/s, done.
Resolving deltas: 100% (89/89), done.


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Model loaded and LoRA applied.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 5.6.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


[transformers] Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
[transformers] Unsloth 2026.4.7 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


Model loaded and LoRA applied.


# Section 2: Environment & Rollout
Define rollout collection using the wildfire env, serializer, and action parser.

In [ ]:
import torch
from env import WildfireEnv
from env.serialization import serialize_observation
from env.action_parser import parse_action
from env.models import TIER_EASY, TIER_MEDIUM, TIER_HARD

TIER_MAX_STEPS = {'easy': 80, 'medium': 150, 'hard': 300}

SYSTEM_PROMPT = (
    'You are an AI Incident Commander managing wildfire containment. '
    'You will receive a situation briefing each step. '
    'Respond with ONLY a valid JSON action object and nothing else. '
    'Example: {"action_type": "idle"}'
)

In [ ]:
def collect_rollout(env, model, tokenizer, tier, seed):
    """Run one episode and return a list of trajectory dicts."""
    max_steps = TIER_MAX_STEPS[tier]
    obs = env.reset(task_id=tier, seed=seed)
    trajectory = []
    done = False
    step = 0

    FastLanguageModel.for_inference(model)

    while not done and step < max_steps:
        prompt_text = serialize_observation(obs, step, max_steps)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt_text},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids, max_new_tokens=128, temperature=0.7,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        completion_ids = output_ids[0][input_ids.shape[1]:]
        completion = tokenizer.decode(completion_ids, skip_special_tokens=True)

        action, parse_status = parse_action(completion, obs)
        result = env.step(action)

        trajectory.append({
            'prompt': prompt_text,
            'completion': completion,
            'reward': result.reward,
            'step_status': parse_status,
            'done': result.done,
        })

        obs = result.observation
        done = result.done
        step += 1

    return trajectory

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # select checkpoint_step_140.zip
os.makedirs('checkpoints/step_140', exist_ok=True)
with zipfile.ZipFile('checkpoint_step_140.zip', 'r') as z:
    z.extractall('checkpoints/step_140')
print('Checkpoint restored.')


from peft import PeftModel
model.load_adapter('checkpoints/step_140', adapter_name="default")
print('Adapter loaded.')

Saving checkpoint_step_140.zip to checkpoint_step_140 (1).zip
Checkpoint restored.
Adapter loaded.


In [ ]:
model.load_adapter('checkpoints/step_140', adapter_name='default')
print('Adapter loaded.')

Adapter loaded.


# Section 3: GRPO Training Loop
Uses TRL GRPOTrainer with a curriculum controller wired in.

In [ ]:
# Resume from checkpoint if one exists
import glob, json

CHECKPOINT_DIR = './checkpoints'
STATS_FILE = './training_stats.json'

latest_ckpt = None
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/step_*'), key=os.path.getmtime)
if ckpts:
    latest_ckpt = ckpts[-1]
    print(f'Resuming from checkpoint: {latest_ckpt}')
    from peft import PeftModel
    model.load_adapter(latest_ckpt)
else:
    print('No checkpoint found â€” training from scratch.')

Resuming from checkpoint: ./checkpoints/step_140


TypeError: PeftModel.load_adapter() missing 1 required positional argument: 'adapter_name'

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from env.curriculum import CurriculumController
from agents.heuristic_agent import HeuristicAgent
import random

controller = CurriculumController(
      start_tier='easy',
      thresholds={'easy': 6.0, 'medium': 5.0},
  )

SEED_POOL = list(range(100))
NUM_GENERATIONS = 8
ROLLOUT_STEPS = 15  # candidate action at step 0, heuristic for steps 1-14


def reward_fn(completions, prompts, tier=None, **kwargs):
    """
    GRPO reward: apply the candidate action as step 0, then run the
    heuristic agent for ROLLOUT_STEPS-1 more steps and return cumulative
    reward.  Using heuristic continuation avoids model inference inside
    reward_fn while still giving a meaningful multi-step signal.
    TRL passes the 'tier' dataset column here so each rollout uses the
    same difficulty tier that generated the prompt.
    """
    rewards = []
    heuristic = HeuristicAgent()

    for i, completion in enumerate(completions):
        ep_tier = tier[i] if tier is not None else controller.get_tier()
        seed = random.choice(SEED_POOL)
        max_steps = TIER_MAX_STEPS[ep_tier]

        mini_env = WildfireEnv()
        obs = mini_env.reset(task_id=ep_tier, seed=seed)
        total_reward = 0.0
        done = False

        # Score the candidate completion as the first action
        completion_text = (
            completion if isinstance(completion, str)
            else completion[0]['content']
        )
        action, _ = parse_action(completion_text, obs)
        result = mini_env.step(action)
        total_reward += result.reward
        obs = result.observation
        done = result.done

        # Continue with heuristic agent for the remaining rollout steps
        for _step in range(1, ROLLOUT_STEPS):
            if done:
                break
            action = heuristic.act(obs)
            result = mini_env.step(action)
            total_reward += result.reward
            obs = result.observation
            done = result.done

        rewards.append(total_reward)

    # Update curriculum based on mean reward of this batch
    mean_reward = sum(rewards) / len(rewards)
    promoted = controller.after_episode(mean_reward)
    if promoted:
        print(f'  *** Curriculum promoted to: {promoted} ***')

    return rewards


grpo_config = GRPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_generations=NUM_GENERATIONS,
    learning_rate=5e-6,
    max_steps=200,
    save_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_completion_length=128,
    logging_steps=1,
)
print('GRPO config ready.')


In [ ]:
from datasets import Dataset

FastLanguageModel.for_training(model)


def build_prompt_dataset(n=50):
    """Build prompts for the current curriculum tier.
    The 'tier' column is forwarded to reward_fn by GRPOTrainer so each
    rollout uses the same difficulty that generated the prompt.
    """
    rows = []
    env_tmp = WildfireEnv()
    tier = controller.get_tier()
    for i in range(n):
        seed = SEED_POOL[i % len(SEED_POOL)]
        obs = env_tmp.reset(task_id=tier, seed=seed)
        max_steps = TIER_MAX_STEPS[tier]
        prompt = serialize_observation(obs, 0, max_steps)
        rows.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt},
            ],
            'tier': tier,
        })
    return rows


dataset = Dataset.from_list(build_prompt_dataset(200))

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=dataset,
)

print(f'Starting GRPO training — {grpo_config.max_steps} steps, {NUM_GENERATIONS} generations/prompt')
print(f'Model: {MODEL_NAME}  |  Start tier: {controller.get_tier()}  |  Rollout steps: {ROLLOUT_STEPS}')
trainer.train(resume_from_checkpoint=latest_ckpt)
print('Training complete.')

# Save training history for the plot cell
history = controller.get_history()
training_stats = [
    {'step': ep_idx, 'tier': t, 'mean_reward': reward}
    for ep_idx, t, reward in history
]
with open(STATS_FILE, 'w') as f:
    json.dump(training_stats, f, indent=2)
print(f'Training history saved -> {STATS_FILE}')


# Section 4: Checkpointing & Recovery
Save final adapter and verify the checkpoint is loadable.

In [ ]:
final_ckpt = f'{CHECKPOINT_DIR}/final'
model.save_pretrained(final_ckpt)
tokenizer.save_pretrained(final_ckpt)
print(f'Final adapter saved -> {final_ckpt}')

# Verify reload
from unsloth import FastLanguageModel as FLM
verify_model, verify_tok = FLM.from_pretrained(
    model_name=final_ckpt,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print('Checkpoint reload verified.')

# Section 5: Plot Reward Curve
Load training stats and plot mean reward vs step with tier promotion markers.

In [ ]:
import matplotlib.pyplot as plt
import json

with open(STATS_FILE) as f:
    stats = json.load(f)

steps = [s['step'] for s in stats]
rewards = [s['mean_reward'] for s in stats]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, rewards, alpha=0.4, color='steelblue', label='Episode reward')

window = 5
if len(rewards) >= window:
    ma = [sum(rewards[max(0,i-window):i+1]) / min(i+1, window) for i in range(len(rewards))]
    ax.plot(steps, ma, color='steelblue', linewidth=2, label=f'MA-{window}')

tier_colors = {'medium': 'orange', 'hard': 'red'}
for ep_idx, new_tier in controller.promotion_log:
    color = tier_colors.get(new_tier, 'gray')
    ax.axvline(x=ep_idx, color=color, linestyle='--', alpha=0.7)
    y_max = max(rewards) if rewards else 1.0
    ax.text(ep_idx + 0.3, y_max * 0.9, new_tier, color=color, fontsize=8)

ax.set_xlabel('Training Step')
ax.set_ylabel('Episode Reward')
ax.set_title('GRPO Training â€” Wildfire Containment Simulator')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=100)
plt.show()
print('Saved reward_curve.png')

In [ ]:
  import os, glob
  CHECKPOINT_DIR = './checkpoints'
  STATS_FILE = './training_stats.json'

  latest_ckpt = None
  ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/step_*'), key=os.path.getmtime)
  if ckpts:
      latest_ckpt = ckpts[-1]
      print(f'Found checkpoint: {latest_ckpt}')
      model.load_adapter(latest_ckpt, adapter_name='default')
      print('Adapter loaded.')
  else:
      print('No checkpoint found - training from scratch.')


Found checkpoint: ./checkpoints/step_140
Adapter loaded.


# Section 6: Evaluate Trained Model vs Baselines
Run the trained model through the same graders used for the heuristic/random
baselines and print a comparison table against `scripts/results.json`.

In [ ]:
import json
import graders.grader_easy as grader_easy
import graders.grader_medium as grader_medium
import graders.grader_hard as grader_hard

EVAL_SEEDS = [42, 43, 44]  # 3 seeds for speed; baselines used 5


class TrainedModelAgent:
    """Grader-compatible agent wrapping the fine-tuned model.
    Graders only call act(); call reset() between episodes to clear
    the step counter.
    """

    def __init__(self, model, tokenizer, max_steps):
        self.model = model
        self.tokenizer = tokenizer
        self.max_steps = max_steps
        self._step = 0

    def reset(self):
        self._step = 0

    def act(self, obs):
        FastLanguageModel.for_inference(self.model)
        prompt_text = serialize_observation(obs, self._step, self.max_steps)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt_text},
        ]
        input_ids = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to(self.model.device)
        with torch.no_grad():
            out_ids = self.model.generate(
                input_ids, max_new_tokens=128, temperature=0.0,
                do_sample=False, pad_token_id=self.tokenizer.eos_token_id
            )
        completion = self.tokenizer.decode(
            out_ids[0][input_ids.shape[1]:], skip_special_tokens=True
        )
        action, _ = parse_action(completion, obs)
        self._step += 1
        return action


GRADER_MAP = {
    'easy':   (grader_easy,   TIER_MAX_STEPS['easy']),
    'medium': (grader_medium, TIER_MAX_STEPS['medium']),
    'hard':   (grader_hard,   TIER_MAX_STEPS['hard']),
}

with open('scripts/results.json') as f:
    baselines = json.load(f)

print('=== Evaluation: Trained Model vs Baselines ===')
print(f'Seeds: {EVAL_SEEDS}  (baselines used seeds 42-46, 5 runs)\n')
print(f'{"Tier":<8} {"Trained":>10} {"Heuristic":>12} {"Random":>10}')
print('-' * 45)

for tier, (grader_mod, max_steps) in GRADER_MAP.items():
    agent = TrainedModelAgent(model, tokenizer, max_steps)
    scores = []
    for seed in EVAL_SEEDS:
        agent.reset()
        score, details = grader_mod.grade(agent, seed=seed)
        scores.append(score)
        print(f'  {tier} seed={seed}: reward={score:.3f}  '
              f'containment={details["containment_pct"]:.1%}  '
              f'pop_saved={details["pop_saved_pct"]:.1%}')
    mean_score = sum(scores) / len(scores)
    heuristic_mean = baselines['heuristic'][tier]['mean']
    random_mean = baselines['random'][tier]['mean']
    print(f'{tier:<8} {mean_score:>10.3f} {heuristic_mean:>12.3f} {random_mean:>10.3f}\n')

print('Done. Beating the heuristic on any tier means the model learned something useful.')


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== Evaluation: Trained Model vs Baselines ===
Seeds: [42, 43, 44]  (baselines used seeds 42-46, 5 runs)

Tier        Trained    Heuristic     Random
---------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local

  easy seed=42: reward=4.875  containment=0.0%  pop_saved=100.0%


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transform

...

  hard seed=42: reward=-7.619  containment=0.0%  pop_saved=96.5%


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

  hard seed=43: reward=-3.919  containment=0.0%  pop_saved=96.5%


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

  hard seed=44: reward=5.300  containment=0.0%  pop_saved=100.0%
hard         -2.079        5.319      5.386

Done. Beating the heuristic on any tier means the model learned something useful.
